# Generate a LangGraph CLI dataset with NeMo Data Designer 0.7

This notebook creates natural-language requests and deterministic structured labels for the current LangGraph CLI. The labels are derived from sampled values rather than generated by a second LLM call, so a fluent request cannot silently acquire a wrong target.

## 1. Configure OpenRouter access

Data Designer uses Nemotron 3 Nano through OpenRouter only to phrase natural-language requests. It disables reasoning traces because they are not training features for this task.

In [ ]:
import getpass
import os

if not os.environ.get("OPENROUTER_API_KEY"):
    os.environ["OPENROUTER_API_KEY"] = getpass.getpass("OpenRouter API key: ")

## 2. Build the Data Designer configuration

Data Designer 0.7 exposes configuration classes through `data_designer.config` and the runtime interface through `data_designer.interface`.

In [ ]:
import data_designer.config as dd
from data_designer.interface import DataDesigner

from tutorial_utils import build_expected_output as make_expected_output

designer = DataDesigner(
    model_providers=[
        dd.ModelProvider(
            name="openrouter",
            endpoint="https://openrouter.ai/api/v1",
            provider_type="openai",
            api_key="OPENROUTER_API_KEY",
        )
    ]
)
model_configs = [
    dd.ModelConfig(
        alias="command-generator",
        provider="openrouter",
        model="nvidia/nemotron-3-nano-30b-a3b",
        inference_parameters=dd.ChatCompletionInferenceParams(
            temperature=0.8,
            top_p=0.95,
            max_tokens=256,
            extra_body={
                "reasoning": {"effort": "none"},
                "provider": {"allow_fallbacks": False},
            },
        ),
    )
]
config_builder = dd.DataDesignerConfigBuilder(model_configs=model_configs)

In [ ]:
config_builder.add_column(
    dd.SamplerColumnConfig(
        name="command",
        sampler_type=dd.SamplerType.CATEGORY,
        params=dd.CategorySamplerParams(values=["new", "dev", "up", "build", "dockerfile"]),
    )
)
config_builder.add_column(
    dd.SamplerColumnConfig(
        name="template",
        sampler_type=dd.SamplerType.CATEGORY,
        params=dd.CategorySamplerParams(
            values=[
                "deep-agent-python",
                "deep-agent-js",
                "agent-python",
                "new-langgraph-project-python",
                "new-langgraph-project-js",
            ]
        ),
    )
)
config_builder.add_column(
    dd.SamplerColumnConfig(
        name="project_path",
        sampler_type=dd.SamplerType.CATEGORY,
        params=dd.CategorySamplerParams(
            values=["agents/demo", "examples/support-agent", "apps/research-agent"]
        ),
    )
)
config_builder.add_column(
    dd.SamplerColumnConfig(
        name="port",
        sampler_type=dd.SamplerType.UNIFORM,
        params=dd.UniformSamplerParams(low=3000, high=9000),
        convert_to="int",
    )
)
for name, probability in (("no_browser", 0.25), ("watch", 0.35), ("wait", 0.5)):
    config_builder.add_column(
        dd.SamplerColumnConfig(
            name=name,
            sampler_type=dd.SamplerType.BERNOULLI,
            params=dd.BernoulliSamplerParams(p=probability),
        )
    )
config_builder.add_column(
    dd.SamplerColumnConfig(
        name="image_tag",
        sampler_type=dd.SamplerType.CATEGORY,
        params=dd.CategorySamplerParams(
            values=["demo-agent:latest", "langgraph-app:v1", "team/research-agent:dev"]
        ),
    )
)
config_builder.add_column(
    dd.SamplerColumnConfig(
        name="dockerfile_path",
        sampler_type=dd.SamplerType.CATEGORY,
        params=dd.CategorySamplerParams(
            values=["Dockerfile", "deploy/Dockerfile", "docker/Dockerfile.langgraph"]
        ),
    )
)

## 3. Create a deterministic target, then phrase the request

The custom column is the source of truth. The LLM sees that exact target and only turns it into a natural request.

In [ ]:
@dd.custom_column_generator(
    required_columns=[
        "command",
        "template",
        "project_path",
        "port",
        "no_browser",
        "watch",
        "wait",
        "image_tag",
        "dockerfile_path",
    ]
)
def expected_output_generator(row: dict) -> dict:
    row["expected_output"] = make_expected_output(row)
    return row


config_builder.add_column(
    dd.CustomColumnConfig(
        name="expected_output",
        generator_function=expected_output_generator,
    )
)
config_builder.add_column(
    dd.LLMTextColumnConfig(
        name="input",
        model_alias="command-generator",
        prompt=(
            "Write one natural, conversational user request for the LangGraph CLI.\n"
            "Its exact intended tool call is: {{ expected_output }}\n\n"
            "Copy every non-null argument value character-for-character into the request. "
            "Do not paraphrase a template ID, path, port, image tag, or Dockerfile path, "
            "and introduce no other flags or operations."
        ),
        system_prompt="Return only the single user-request sentence.",
    )
)
designer.validate(config_builder)

## 4. Preview before spending a full generation budget

In [ ]:
preview = designer.preview(config_builder=config_builder, num_records=5)
preview.display_sample_record()
preview.dataset[["input", "expected_output"]]

Inspect all five rows. In particular, confirm that every phrase mentions all non-null values in `expected_output`. Refine the wording prompt before continuing if it does not.

In [ ]:
result = designer.create(
    config_builder=config_builder,
    num_records=250,
    dataset_name="langgraph_cli",
)
dataset_df = result.load_dataset()
from tutorial_utils import normalize_expected_output, repair_generated_request
dataset_df["expected_output"] = dataset_df["expected_output"].map(normalize_expected_output)
original_inputs = dataset_df["input"].astype(str)
dataset_df["input"] = [
    repair_generated_request(user_input, expected_output)
    for user_input, expected_output in zip(
        original_inputs, dataset_df["expected_output"], strict=True
    )
]
repair_count = sum(
    original.strip() != repaired
    for original, repaired in zip(original_inputs, dataset_df["input"], strict=True)
)
print(f"Replaced {repair_count} incomplete generated requests with deterministic fallbacks.")
dataset_df[["input", "expected_output"]].head()

## 5. Audit the completed generation

The preview validates the configuration, but the full generation contains new model-written requests. Every deterministic target is checked against the runtime grammar below, then a reproducible 25-row sample is shown for semantic review. Confirm that each request mentions every non-null argument and introduces no extra intent. For a production dataset, use a larger independent human or judge-model audit.

In [ ]:
import json
import tempfile

from IPython.display import display

from bash_agent.commands import invocation_from_payload

with tempfile.TemporaryDirectory() as validation_root:
    for index, row in dataset_df.iterrows():
        target = row["expected_output"]
        if isinstance(target, str):
            target = json.loads(target)
        invocation_from_payload(target, validation_root)
        if not str(row["input"]).strip():
            raise ValueError(f"Generated request {index} is empty")

audit_df = dataset_df.sample(n=min(25, len(dataset_df)), random_state=42)
display(audit_df[["input", "expected_output"]])
if input("Type APPROVE after reviewing the audit sample: ").strip() != "APPROVE":
    raise RuntimeError("Dataset export cancelled; revise the generation prompt and rerun.")

## 6. Export NeMo Gym 0.3 records

Gym datasets use `responses_create_params` plus task-specific verifier fields. The same records are consumed by the GRPO notebook.

In [ ]:
import json
from pathlib import Path

from sklearn.model_selection import train_test_split

from tutorial_utils import build_gym_record

train_df, validation_df = train_test_split(
    dataset_df,
    test_size=0.1,
    random_state=42,
    stratify=dataset_df["command"],
)
output_dir = Path("data/langgraph_cli")
output_dir.mkdir(parents=True, exist_ok=True)


def save_jsonl(frame, path: Path) -> None:
    with path.open("w", encoding="utf-8") as stream:
        for _, row in frame.iterrows():
            record = build_gym_record(str(row["input"]), row["expected_output"])
            stream.write(json.dumps(record, ensure_ascii=False) + "\n")


save_jsonl(train_df, output_dir / "train.jsonl")
save_jsonl(validation_df, output_dir / "validation.jsonl")
print(f"Wrote {len(train_df)} train and {len(validation_df)} validation records to {output_dir}")